# 09 — Evaluation of LLM and rule-based extraction

Two independent evaluations against the gold file:

| Source | Fields it produces |
|---|---|
| **LLM (notebook 07)** | title, authors, doi, publication_year, document_type, language, keywords |
| **Rules (notebook 08)** | doi, document_type |

Each field is scored with the metric that fits it:
- title, doi, publication_year, document_type, language → exact match (title uses ≥ 0.90 character similarity)
- authors, keywords → F1 per document, averaged

A field the method does not produce, or that has no gold labels, is reported as such — not as 0.00.

In [42]:
from pathlib import Path
import pandas as pd
from IPython.display import display

OUT_DIR = Path('../output/extraction').resolve()

llm   = pd.read_csv(OUT_DIR / 'eurollm_extractions.csv', dtype=str, keep_default_na=False)
rules = pd.read_csv(OUT_DIR / 'doc_features.csv',        dtype=str, keep_default_na=False)

rules = rules[['document_id', 'doi', 'document_type_rule', 'doi_evidence', 'doi_score']]
merged = llm.merge(rules, on='document_id', how='left', suffixes=('', '_rule'))

print(f'{len(merged)} documents')
print(f'LLM columns  : {list(llm.columns)}')
print(f'Rule columns : doi, document_type_rule, doi_evidence, doi_score')
print(f'Rule predictions: {len(rules)}')

25 documents
LLM columns  : ['document_id', 'file_name', 'parse_ok', 'title', 'authors', 'doi', 'publication_year', 'document_type', 'language', 'keywords']
Rule columns : doi, document_type_rule, doi_evidence, doi_score
Rule predictions: 92


In [43]:
for row in merged.itertuples(index=False):
    print('=' * 100)
    print(row.file_name)
    print('-' * 100)

    print(f'{"title":18s}  LLM : {row.title}')
    print(f'{"authors":18s}  LLM : {row.authors}')
    print(f'{"keywords":18s}  LLM : {row.keywords}')
    print(f'{"year":18s}  LLM : {row.publication_year}')
    print(f'{"type":18s}  LLM : {row.document_type}')
    print(f'{"language":18s}  LLM : {row.language}')
    print()
    print(f'{"doi":18s}  LLM : {row.doi}')
    print(f'{"":18s}  RULE: {row.doi_rule}')
    if row.doi_evidence_rule if hasattr(row, "doi_evidence_rule") else False:
        print(f'{"":18s}        evidence: {row.doi_evidence_rule}  (score {row.doi_score_rule})')
    print()
    print(f'{"type":18s}  RULE: {row.document_type_rule}')
    print()

001_adaptive-capacities-for-diversified-flood-risk-management-strategies-learning-from-pilot-projects.pdf
----------------------------------------------------------------------------------------------------
title               LLM : Adaptive Capacities for Diversified Flood Risk Management
authors             LLM : Cosoveanu, Flavia Simona; Buijs, Jean-Marie; Bakker, Marloes; Terpstra, Teun
keywords            LLM : 
year                LLM : 2019
type                LLM : conference_paper
language            LLM : en

doi                 LLM : 
                    RULE: 10.3390/w11122643

type                RULE: journal_article

005_building-relationships-in-higher-education-to-support-students-motivation.pdf
----------------------------------------------------------------------------------------------------
title               LLM : Building relationships in higher education to support students' motivation
authors             LLM : Martijn J. M. Leenknecht; Ingrid Snijders; Lisette

In [47]:
view = merged[[
    'file_name',
    'title', 'authors', 'keywords', 'doi_rule',
    'publication_year', 'document_type', 'document_type_rule', 'language',
]].rename(columns={
    'doi_rule': 'doi (rule)',
    'document_type_rule': 'type (rule)',
})

with pd.option_context('display.max_colwidth', 60):
    display(view)

,file_name,title,authors,keywords,doi (rule),publication_year,document_type,type (rule),language
0,001_adaptive-capacities-for-diversified-flood-risk-manag...,Adaptive Capacities for Diversified Flood Risk Management,"Cosoveanu, Flavia Simona; Buijs, Jean-Marie; Bakker, Mar...",,10.3390/w11122643,2019,conference_paper,journal_article,en
1,005_building-relationships-in-higher-education-to-suppor...,Building relationships in higher education to support st...,Martijn J. M. Leenknecht; Ingrid Snijders; Lisette Wijni...,,10.1080/13562517.2020.1839748,2023,journal_article,journal_article,en
2,011_fine-tuning-the-flavor-of-tetraselmis-chuii-by-modif...,Fine-tuning the flavor of Tetraselmis chuii by modifying...,Bert Coleman; Christof Van Poucke; Bavo De Witte; Valent...,algal research; Tetraselmis chuii; nitrogen starvation; ...,10.1016/j.algal.2023.103208,2023,journal_article,journal_article,en
3,013_grip-op-teksten-grip-op-kennis.pdf,"Grip on Texts, Grip on Knowledge",Daniela Polišenská; Joep Schins; Helge Bonset,,10.1080/0305569022000003681,2025,conference_paper,journal_article,en
4,016_impacts-of-counting-protocols-for-light-microscopy-o...,Impacts of counting protocols for light microscopy on es...,Pascalle Jacobs; Léon Serre-Fredj; Reinoud P. T. Koeman;...,,10.1002/lom3.10651,2024,conference_paper,journal_article,en
5,019_lezen-en-schrijven-in-natuur-en-techniekonderwijs.pdf,Lezen en schrijven in Natuur- en Techniekonderwijs: Teks...,Hilde Kooiker-den Boer; Ted Sanders; Jacqueline Evers-Ve...,,10.5117/tvt2025.01.002.kooi,2025,conference_paper,journal_article,nl
6,023_relationship-quality-in-higher-education-and-the-int...,Relationship Quality in Higher Education and the Interpl...,Ingrid Snijders; Lisette Wijnia; Rebecca M. Kuiper; Remy...,,10.1111/bjep.12455,2021,journal_article,journal_article,en
7,027_the-use-of-tetraselmis-chuii-as-seafood-flavoring-ag...,The Use of Tetraselmis chuii as Seafood Flavoring Agent ...,Bert Coelho; Christof Van Poucke; Bavo De Witte; Valenti...,microalgae; Tetraselmis chuii; seafood alternative; sens...,10.1016/j.algal.2024.103538,2024,journal_article,journal_article,en
8,029_transition-towards-integrated-flood-risk-management-...,Transition Towards Integrated Flood Risk Management Stra...,Jean-Marie Buijs; Flavia Simona Cosoveanu; Marloes Bakke...,,,2020,conference_paper,other,en
9,031_vakdidactische-kennisclips-over-tekststructuur-een-v...,Vakdidactische kennisclips over tekststructuur,H.S. Kooiker-den Boer; S.T.M. Bogaerds-Hazenberg; T.J.M....,,,2024,conference_paper,other,en


In [ ]:
import pymupdf
from pathlib import Path

PDF_DIR = Path('../pdfs').resolve()

for paper in ['001_adaptive-capacities-for-diversified-flood-risk-management-strategies-learning-from-pilot-projects.pdf',
              '005_building-relationships-in-higher-education-to-support-students-motivation.pdf',
              '011_fine-tuning-the-flavor-of-tetraselmis-chuii-by-modifying-nitrogen-supply.pdf']:

    with pymupdf.open(PDF_DIR / paper) as pdf:
        text = '\n'.join(p.get_text() for p in pdf)

    i = text.lower().find('keywords')
    if i == -1:
        print(f'{paper[:40]:40s}  NO "keywords" in text')
        continue

    print(f'{paper[:40]:40s}')
    print(f'  Context: {repr(text[max(0,i-40):i+200])}')
    print()

001_adaptive-capacities-for-diversified-
  Context: 'ctions for improvement are highlighted.\nKeywords: adaptive capacities; diversiﬁed ﬂood risk management strategies; pilot project; governance\nnetworks; learning\n1. Introduction\nClimate change will result in increased exposure of low-lying co'

005_building-relationships-in-higher-edu
  Context: 'ed 27 June 2020\nAccepted 5 October 2020\nKEYWORDS\nStudent-staﬀinteractions;\nneed-supportive teaching;\nrelationship quality;\nmotivation; self-\ndetermination theory\nHigher education (HE) is diﬀerent from other educational contexts in several w'

011_fine-tuning-the-flavor-of-tetraselmi
  Context: ' Netherlands   \nA R T I C L E I N F O  \nKeywords: \nMicroalgae \nTetraselmis chuii \nNitrogen starvation \nSensory evaluation \nVolatile organic compounds \nUmami \nA B S T R A C T  \nDried Tetraselmis chuii biomass has potential as flavoring agent'



In [ ]:
from pathlib import Path
import pymupdf

PDF_DIR = Path('../pdfs').resolve()

for paper in ['001_adaptive-capacities-for-diversified-flood-risk-management-strategies-learning-from-pilot-projects.pdf',
              '005_building-relationships-in-higher-education-to-support-students-motivation.pdf']:

    with pymupdf.open(PDF_DIR / paper) as pdf:
        page1 = pdf[0].get_text()

    pos = page1.lower().find('keywords')
    print(f'{paper[:40]}')
    print(f'  Position of "Keywords" on page 1: char {pos}  (prompt uses first ~4500)')
    print(f'  Inside prompt: {pos >= 0 and pos < 4500}')
    print()

001_adaptive-capacities-for-diversified-
  Position of "Keywords" on page 1: char 2005  (prompt uses first ~4500)
  Inside prompt: True

005_building-relationships-in-higher-edu
  Position of "Keywords" on page 1: char -1  (prompt uses first ~4500)
  Inside prompt: False

